# PID / flight-purity cut diagnostics — mass-peak visualization

Companion to `03_pid_optimization.ipynb`, not a new pipeline stage.
`run_stage03.py`'s Punzi FOM picks the PID selectors numerically (efficiency
and background counted directly in the mES/$\Delta E$ signal region); this
notebook instead shows **the composite's own invariant-mass distribution,
before and after a cut, with the S (peak) window and its contiguous
sideband bands shaded** -- the same peak/sideband convention used by Stage
2's $S/\sqrt{S+B}$ scans (`channel_config.FOM_SIDEBAND_WIDTH_MULT`). This is
a visual cross-check that a cut removes sideband-like (background-like)
candidates preferentially over peak candidates, independent of whichever
FOM/method actually picked the cut -- mirroring the diagnostic style used in
the $\bar{p}\Lambda^0$ analysis's `Lambda_purity_studies.ipynb`.

Two sections:

1. **$\Lambda_c^+$ per-mode PID diagnostics** (`Lam0LamC` only): for each of
   the 4 decay modes, the $\Lambda_c^+$ mass before vs. after that mode's
   recommended PID selector (`channel_config.py`'s `pid` section, applied at
   the Stage 3 checkpoint), shown separately for signal MC and
   luminosity-weighted background MC.
2. **$\Lambda^0$ flight-significance and PID diagnostics** (either channel):
   the $\Lambda^0$ mass before vs. after (a) the current Stage 2
   flight-significance cut, and (b) the Stage 3 PID selector applied on top
   of it -- again signal MC and background MC side by side.

MC only, no collision data.

In [ ]:
import sys
sys.path.insert(0, '..')

get_ipython().run_line_magic('load_ext', 'autoreload')
get_ipython().run_line_magic('autoreload', '2')

import numpy as np
import awkward as ak
import matplotlib.pylab as plt

from channel_config import get_channel_config, BACKGROUND_SP_MODES, SIGNAL_SP_MODE, FOM_SIDEBAND_WIDTH_MULT
import datasets
import cutflow
import plotting

In [ ]:
# Select the channel here: 'Lam0Lam0' or 'Lam0LamC'
CHANNEL = 'Lam0Lam0'

config = get_channel_config(CHANNEL)
print(f"Channel: {config['name']}   {config['decay_label']}")

In [ ]:
data_sp, _ = datasets.load_datasets(CHANNEL, sp_or_data='sp')
datasets.add_derived_fields(data_sp, config)

weights = datasets.get_scaling_weights(BACKGROUND_SP_MODES)

print(f"MC (SP) events: {len(data_sp)}")

In [ ]:
# Flatten a composite's mass under a jagged candidate mask.
def flat_mass(data, mass_var, mask):
    return ak.to_numpy(ak.flatten(data[mass_var][mask], axis=None))

# Same, for luminosity-weighted background MC: one entry per candidate,
# repeating that SP mode's scaling weight.
def flat_mass_and_bkg_weights(data_sp, mass_var, mask, weights):
    parts_m, parts_w = [], []
    for spmode in BACKGROUND_SP_MODES:
        sel = (data_sp['spmode'] == spmode) & mask
        m = flat_mass(data_sp, mass_var, sel)
        if len(m) == 0:
            continue
        parts_m.append(m)
        parts_w.append(np.full(len(m), weights[str(spmode)]))
    if not parts_m:
        return np.array([]), np.array([])
    return np.concatenate(parts_m), np.concatenate(parts_w)

## $\Lambda_c^+$ per-mode PID diagnostics (`Lam0LamC` only)

"Before" = mode selected + $K_S^0$ gate applied (the purity cuts unrelated
to PID); "after" = + that mode's recommended PID selector
(`channel_config.py`, applied 2026-07-25). Neither is mass-windowed -- the
window is only drawn as the shaded S region.

In [ ]:
if CHANNEL == 'Lam0LamC':
    mass_var = config['composites']['LambdaC']['mass_var']
    hist_def = config['hist_defs']['LambdaC_unc_Mass']
    windows = config['composites']['LambdaC']['mass_windows_per_mode']

    mode_arr = cutflow.get_lambdac_decay_mode(data_sp)
    k0s_gate = cutflow.get_lambdac_k0s_gate(data_sp, config)

    lambda0_sel = config['pid']['lambda0_selector']
    lambda0_pid_mask = cutflow.get_lambda0_pid_mask(
        data_sp, p_selector=lambda0_sel['p'], pi_selector=lambda0_sel['pi'])
    lamc_pid_mask = cutflow.get_lambdac_pid_mask(
        data_sp, config['pid']['lambdac_selector_per_mode'], lambda0_pid_mask)

    mask_sig = data_sp['spmode'] == SIGNAL_SP_MODE

    fig, axes = plt.subplots(4, 2, figsize=(11, 15))

    for i, m in enumerate(sorted(config['lambdac_modes'].keys())):
        before = (mode_arr == m) & k0s_gate
        after = before & lamc_pid_mask
        sel_str = ", ".join(f"{p}:{n.split('KM')[0]}" for p, n in
                            config['pid']['lambdac_selector_per_mode'].get(m, {}).items())

        m_sig_before = flat_mass(data_sp, mass_var, before & mask_sig)
        m_sig_after = flat_mass(data_sp, mass_var, after & mask_sig)
        plotting.plot_mass_cut_diagnostic(
            m_sig_before, m_sig_after, windows[m], FOM_SIDEBAND_WIDTH_MULT, hist_def=hist_def,
            label_before='before PID', label_after=f'after PID ({sel_str})',
            title=f"mode {m} ({config['lambdac_modes'][m]}): signal MC", ax=axes[i][0])

        m_bkg_before, w_bkg_before = flat_mass_and_bkg_weights(data_sp, mass_var, before, weights)
        m_bkg_after, w_bkg_after = flat_mass_and_bkg_weights(data_sp, mass_var, after, weights)
        plotting.plot_mass_cut_diagnostic(
            m_bkg_before, m_bkg_after, windows[m], FOM_SIDEBAND_WIDTH_MULT, hist_def=hist_def,
            weights_before=w_bkg_before, weights_after=w_bkg_after,
            label_before='before PID', label_after=f'after PID ({sel_str})',
            title=f"mode {m} ({config['lambdac_modes'][m]}): background MC (wtd.)", ax=axes[i][1])

    plt.tight_layout()
    plt.savefig(f"{plotting.plot_dir(config)}/lambdac_mode_pid_mass_diagnostic.png", dpi=150)
else:
    print(f"Skipped -- LambdaC section is Lam0LamC-only (CHANNEL={CHANNEL})")

## $\Lambda^0$ flight-significance diagnostic (either channel)

"Before" = no flight cut at all (raw $\Lambda^0$ candidates); "after" = the
current `channel_config.py` flight-significance cut.

In [ ]:
comp_l0 = config['composites']['Lambda0']
mass_var_l0 = comp_l0['mass_var']
hist_def_l0 = config['hist_defs']['Lambda0_unc_Mass']
window_l0 = comp_l0['mass_window']
flight_var, flight_cut = comp_l0['flight_var'], comp_l0['flight_cut']

mask_sig = data_sp['spmode'] == SIGNAL_SP_MODE

before_flight = ak.ones_like(data_sp[flight_var], dtype=bool)
after_flight = data_sp[flight_var] > flight_cut

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

m_sig_before = flat_mass(data_sp, mass_var_l0, before_flight & mask_sig)
m_sig_after = flat_mass(data_sp, mass_var_l0, after_flight & mask_sig)
plotting.plot_mass_cut_diagnostic(
    m_sig_before, m_sig_after, window_l0, FOM_SIDEBAND_WIDTH_MULT, hist_def=hist_def_l0,
    label_before='no flight cut', label_after=f'flight sig. > {flight_cut:.0f}',
    title=f'{CHANNEL}: signal MC', ax=axes[0])

m_bkg_before, w_bkg_before = flat_mass_and_bkg_weights(data_sp, mass_var_l0, before_flight, weights)
m_bkg_after, w_bkg_after = flat_mass_and_bkg_weights(data_sp, mass_var_l0, after_flight, weights)
plotting.plot_mass_cut_diagnostic(
    m_bkg_before, m_bkg_after, window_l0, FOM_SIDEBAND_WIDTH_MULT, hist_def=hist_def_l0,
    weights_before=w_bkg_before, weights_after=w_bkg_after,
    label_before='no flight cut', label_after=f'flight sig. > {flight_cut:.0f}',
    title=f'{CHANNEL}: background MC (wtd.)', ax=axes[1])

plt.tight_layout()
plt.savefig(f"{plotting.plot_dir(config)}/lambda0_flight_mass_diagnostic.png", dpi=150)

## $\Lambda^0$ PID diagnostic (either channel)

On top of the flight cut above: "before" = flight cut only (Stage 2
baseline); "after" = + the Stage 3 recommended $\Lambda^0$ PID selector.
This is the same marginal comparison the Punzi FOM itself is built on
(`pid_optimization.evaluate_combo`), just shown as a mass distribution
instead of a single efficiency/background number.

In [ ]:
lambda0_sel = config['pid']['lambda0_selector']
lambda0_pid_mask = cutflow.get_lambda0_pid_mask(
    data_sp, p_selector=lambda0_sel['p'], pi_selector=lambda0_sel['pi'])

before_pid = after_flight
after_pid = after_flight & lambda0_pid_mask
sel_str = f"p:{lambda0_sel['p'].split('KM')[0]}, pi:{lambda0_sel['pi'].split('KM')[0]}"

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

m_sig_before = flat_mass(data_sp, mass_var_l0, before_pid & mask_sig)
m_sig_after = flat_mass(data_sp, mass_var_l0, after_pid & mask_sig)
plotting.plot_mass_cut_diagnostic(
    m_sig_before, m_sig_after, window_l0, FOM_SIDEBAND_WIDTH_MULT, hist_def=hist_def_l0,
    label_before='flight cut only', label_after=f'+ PID ({sel_str})',
    title=f'{CHANNEL}: signal MC', ax=axes[0])

m_bkg_before, w_bkg_before = flat_mass_and_bkg_weights(data_sp, mass_var_l0, before_pid, weights)
m_bkg_after, w_bkg_after = flat_mass_and_bkg_weights(data_sp, mass_var_l0, after_pid, weights)
plotting.plot_mass_cut_diagnostic(
    m_bkg_before, m_bkg_after, window_l0, FOM_SIDEBAND_WIDTH_MULT, hist_def=hist_def_l0,
    weights_before=w_bkg_before, weights_after=w_bkg_after,
    label_before='flight cut only', label_after=f'+ PID ({sel_str})',
    title=f'{CHANNEL}: background MC (wtd.)', ax=axes[1])

plt.tight_layout()
plt.savefig(f"{plotting.plot_dir(config)}/lambda0_pid_mass_diagnostic.png", dpi=150)

## Observations

*(fill in after running for both channels)*

- [ ] Do the $\Lambda_c^+$ per-mode PID panels show the background histogram
  shrinking more in the sidebands than in the peak (i.e. the cut is really
  improving purity, not just scaling everything down)?
- [ ] Does the $\Lambda^0$ flight-cut panel show the expected combinatorial
  background shape outside the peak, and does the cut visibly flatten/
  remove it?
- [ ] For $\Lambda^0$'s PID panel (on top of the flight cut): given Stage 3's
  boundary-hugging result (loosest KM rung recommended), does the visual
  before/after difference look proportionate to the small FOM gain, or
  larger/smaller than expected?
- [ ] Re-run with `CHANNEL = 'Lam0Lam0'` and repeat the $\Lambda^0$-section
  checks for that channel.